In [1]:
query = '''
        WITH new_users AS (
        SELECT user_id, create_date as user_join_date
        FROM vod."user" u
        WHERE NOT EXISTS (
            SELECT 1 FROM vod.payments p
            WHERE p.user_id = u.user_id AND p.payment_status = 'SUCCESS'
        )
    ),
    reach AS (
      SELECT e.user_id,
             bool_or(e.event_type = 'visit')         AS visited,
             bool_or(e.event_type = 'click')         AS clicked,
             bool_or(e.event_type = 'add_to_basket') AS basketed,
             bool_or(e.event_type = 'payment')       AS attempted
      FROM vod.events e
      JOIN new_users nu ON nu.user_id = e.user_id
      GROUP BY e.user_id
    ),
    classified AS (
      SELECT nu.user_id,
        CASE
          WHEN r.user_id IS NULL THEN 'S0 invisible (no events)'
          WHEN r.attempted THEN 'S4 failed-payment (tried to pay)'
          WHEN r.basketed THEN 'S3 cart-abandoner (basket, no pay)'
          WHEN r.clicked THEN 'S2 browser (clicked, no basket)'
          ELSE 'S1 visit-only'
        END AS segment
      FROM new_users as nu
      LEFT JOIN reach r ON r.user_id = nu.user_id
    )
    SELECT *
    FROM classified
    ORDER BY segment
'''

In [2]:
import pandas as pd
import sys
sys.path.extend(['/home/jovyan/modules'])

from db_connector import *

user_segments_df = pd.read_sql_query(query,bootcamp_db)
user_segments_df

,user_id,segment
0,bca1ce73-8bc3-465f-b523-9dbe6988b9d4,S0 invisible (no events)
1,32316369-88cf-4260-a216-bbbbf34e81a6,S0 invisible (no events)
2,6335e5b3-4980-4dcf-a892-3bcf774a4bf2,S0 invisible (no events)
3,cd69993f-7997-4e5a-91d0-f599ace5de99,S0 invisible (no events)
4,0653a9b8-b4bd-4a3d-a031-fb92bd33d6f7,S0 invisible (no events)
...,...,...
267822,9454f854-e185-4f46-b0bf-150fc4d89401,S4 failed-payment (tried to pay)
267823,e78c6993-cac1-40ae-819e-fd1749bc9c77,S4 failed-payment (tried to pay)
267824,c905a45c-6cfd-424a-956f-13918942b7a4,S4 failed-payment (tried to pay)
267825,abb167ce-7cd3-4a90-be2f-265dc9969dff,S4 failed-payment (tried to pay)


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

df  = user_segments_df.copy()
# تقسیم به دو گروه ۵۰-۵۰ با حفظ نسبت سگمنت‌ها
df['group'] = None

indices = df.index.values

idx_A, idx_B = train_test_split(
    indices,
    test_size=0.5,           # ۵۰٪ برای گروه B
    random_state=42,         # برای تکرارپذیری
    stratify=df['segment']   
)

df.loc[idx_A, 'group'] = 'Test'
df.loc[idx_B, 'group'] = 'Control'

print(df.groupby(['segment', 'group']).size().unstack(fill_value=0))


group                               Control   Test
segment                                           
S0 invisible (no events)              21349  21349
S1 visit-only                          2776   2775
S2 browser (clicked, no basket)       12018  12018
S3 cart-abandoner (basket, no pay)    82715  82715
S4 failed-payment (tried to pay)      15056  15056


In [12]:
df

,user_id,segment,group
0,bca1ce73-8bc3-465f-b523-9dbe6988b9d4,S0 invisible (no events),Test
1,32316369-88cf-4260-a216-bbbbf34e81a6,S0 invisible (no events),Test
2,6335e5b3-4980-4dcf-a892-3bcf774a4bf2,S0 invisible (no events),Control
3,cd69993f-7997-4e5a-91d0-f599ace5de99,S0 invisible (no events),Test
4,0653a9b8-b4bd-4a3d-a031-fb92bd33d6f7,S0 invisible (no events),Test
...,...,...,...
267822,9454f854-e185-4f46-b0bf-150fc4d89401,S4 failed-payment (tried to pay),Test
267823,e78c6993-cac1-40ae-819e-fd1749bc9c77,S4 failed-payment (tried to pay),Test
267824,c905a45c-6cfd-424a-956f-13918942b7a4,S4 failed-payment (tried to pay),Test
267825,abb167ce-7cd3-4a90-be2f-265dc9969dff,S4 failed-payment (tried to pay),Control


In [ ]:
df.